## Evaluation (memory-safe retrieval on NCVR)

This notebook evaluates retrieval without persisting new files or loading everything into memory. It uses micro-sharded candidate encoding and top-K tracking across shards.

Metrics: Recall@1/10/50, MRR@10, and PR/F1 via threshold sweep on top-1 cosine similarity.


In [13]:
from eval_utils import EvalParams, evaluate_box

# Choose boxes: e.g., 8 for validation, 9 for test
TEST_BOX = 9

In [23]:
MODEL = "nreimers/MiniLM-L6-H384-uncased"    # this was the original model
#MODEL = "outputs/run_minilm_box0_gpu/final"  # or "outputs/run_minilm_box0_gpu/2000"
DEVICE = "cuda"  # or None/"cpu"
# Keep/adjust the other knobs as you prefer
# Optional overrides
DATA_DIR = None  # defaults to data/north_carolina_voters
DEVICE = None    # 'cuda' or 'cpu'; auto-detect if None

# CPU-friendly knobs
BATCH_SIZE = 32
CANDIDATE_CHUNK_SIZE = 20_000
MAX_LENGTH = 64
MAX_QUERIES = 2_000
MAX_CANDIDATES = 100_000
SHARD_MODULUS = 10
SHARD_REMAINDER = 0
SOURCES_TO_EVAL = None  # e.g., ["dataset_1.csv"] to limit or None for no limit


params_test = EvalParams(
    box_id=TEST_BOX,
    model_name_or_path=MODEL,
    data_dir=DATA_DIR,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    candidate_chunk_size=CANDIDATE_CHUNK_SIZE,
    max_length=MAX_LENGTH,
    max_queries_per_source=MAX_QUERIES,
    max_candidates_per_source=MAX_CANDIDATES,
    shard_modulus=SHARD_MODULUS,
    shard_remainder=SHARD_REMAINDER,
    sources_to_eval=SOURCES_TO_EVAL,
    compute_hard_metrics=True,
    classify_top_k=16
)

params_val, params_test


(EvalParams(box_id=8, model_name_or_path='nreimers/MiniLM-L6-H384-uncased', recid_column='recid', entity_columns=('givenname', 'surname', 'postcode', 'suburb'), data_dir=None, device=None, batch_size=32, candidate_chunk_size=20000, ks=(1, 10, 50), verbose=True, log_every_batches=50, max_queries_per_source=2000, max_candidates_per_source=100000, shard_modulus=10, shard_remainder=0, sources_to_eval=None, seed=42, max_length=64, compute_hard_metrics=True, classify_top_k=16),
 EvalParams(box_id=9, model_name_or_path='nreimers/MiniLM-L6-H384-uncased', recid_column='recid', entity_columns=('givenname', 'surname', 'postcode', 'suburb'), data_dir=None, device=None, batch_size=32, candidate_chunk_size=20000, ks=(1, 10, 50), verbose=True, log_every_batches=50, max_queries_per_source=2000, max_candidates_per_source=100000, shard_modulus=10, shard_remainder=0, sources_to_eval=None, seed=42, max_length=64, compute_hard_metrics=True, classify_top_k=16))

In [24]:
# Evaluate on test box (optional)
metrics_test = evaluate_box(params_test)
metrics_test


[eval] Loading and serializing box=9 from /home/nicolas/Documents/record_linkage/data/north_carolina_voters...
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_1.csv: 9,972 rows serialized
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_2.csv: 10,055 rows serialized
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_3.csv: 9,946 rows serialized
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_4.csv: 9,912 rows serialized
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_5.csv: 9,918 rows serialized
[eval] Loaded model 'nreimers/MiniLM-L6-H384-uncased' on device=cpu
[eval] Sources (selected): ['dataset_1.csv', 'dataset_2.csv', 'dataset_3.csv', 'dataset_4.csv', 'dataset_5.csv']
[eval] Row counts per source (selected): {'dataset_1.csv': 9972, 'dataset_2.csv': 10055, 'dataset_3.csv': 9946, 'dataset_4.csv': 9912, 'dataset_5.csv': 9918}
[eval] Encoding queries from dataset_1.csv: 

KeyboardInterrupt: 

In [22]:
# Evaluate on test box (optional)
metrics_test = evaluate_box(params_test)
metrics_test


[eval] Loading and serializing box=9 from /home/nicolas/Documents/record_linkage/data/north_carolina_voters...
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_1.csv: 9,972 rows serialized
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_2.csv: 10,055 rows serialized
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_3.csv: 9,946 rows serialized
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_4.csv: 9,912 rows serialized
[eval]  -> Applying shard filter: recid % 10 == 0
[eval] Source dataset_5.csv: 9,918 rows serialized
[eval] Loaded model 'outputs/run_minilm_box0_gpu/final' on device=cpu
[eval] Sources (selected): ['dataset_1.csv', 'dataset_2.csv', 'dataset_3.csv', 'dataset_4.csv', 'dataset_5.csv']
[eval] Row counts per source (selected): {'dataset_1.csv': 9972, 'dataset_2.csv': 10055, 'dataset_3.csv': 9946, 'dataset_4.csv': 9912, 'dataset_5.csv': 9918}
[eval] Encoding queries from dataset_1.csv

KeyboardInterrupt: 